In [1]:
from datasets import load_dataset
from transformers import Trainer, AutoTokenizer, TrainingArguments, AutoModelForMaskedLM, DataCollatorForLanguageModeling
import torch

In [2]:
checkpoint = "bert-base-cased"

ds = load_dataset('AndyChiang/cloth')

README.md:   0%|          | 0.00/670 [00:00<?, ?B/s]

CLOTH_train_cleaned.json:   0%|          | 0.00/15.6M [00:00<?, ?B/s]

CLOTH_valid_cleaned.json: 0.00B [00:00, ?B/s]

CLOTH_test_cleaned.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/76850 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11067 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11516 [00:00<?, ? examples/s]

In [3]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForMaskedLM.from_pretrained(checkpoint).to("cuda")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
def tokenize_fn(data):
    return tokenizer(data["sentence"], data["answer"],  truncation=True)

In [5]:
tokenized_data = ds.map(tokenize_fn, batched=True)

Map:   0%|          | 0/76850 [00:00<?, ? examples/s]

Map:   0%|          | 0/11067 [00:00<?, ? examples/s]

Map:   0%|          | 0/11516 [00:00<?, ? examples/s]

In [6]:
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer)

In [7]:
training_args = TrainingArguments(
     output_dir="mask_model",
     per_device_train_batch_size=32,
     num_train_epochs=3,
     fp16=True,
     gradient_accumulation_steps=2
)

In [8]:
trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["validation"],
    data_collator=collator,
    processing_class=tokenizer    
)

In [9]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
500,8.271745
1000,7.559537
1500,7.313891


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1803, training_loss=7.616772203662299, metrics={'train_runtime': 3612.5365, 'train_samples_per_second': 63.819, 'train_steps_per_second': 0.499, 'total_flos': 1.5426435887786352e+16, 'train_loss': 7.616772203662299, 'epoch': 3.0})

In [10]:
tokenizer.save_pretrained("./bert_mlm")
model.save_pretrained("./bert_mlm")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
@torch.no_grad()
def predict_mask(model, tokenizer, text, top_k=5):
    # Tokenize input
    inputs = tokenizer(text, return_tensors='pt').to("cuda")

    # Find masked token position
    masked_index = torch.where(inputs['input_ids'][0] == tokenizer.mask_token_id)[0]

    if len(masked_index) == 0:
        return "No mask token found in text"


    outputs = model(**inputs)
    predictions = outputs.logits[0, masked_index]

    # Get top predictions
    top_preds = torch.topk(predictions, top_k, dim=1)

    results = []
    for i in range(top_k):
        token_id = top_preds.indices[0, i].item()
        token = tokenizer.decode([token_id])
        probability = torch.softmax(predictions, dim=-1)[0, token_id].item()
        results.append((token, probability))

    return results

In [12]:
predictions = predict_mask(
  model,
  tokenizer,
  text="The [MASK] of France is Paris",
  top_k=5
)

for token, prob in predictions:
    print(f"{token}: {prob:.3f}")

capital: 0.988
language: 0.002
name: 0.001
city: 0.001
heart: 0.001


In [13]:
predictions = predict_mask(
  model,
  tokenizer,
  text="The [MASK] was not what she expected, leading to much confusion.",
  top_k=5
)

for token, prob in predictions:
    print(f"{token}: {prob:.3f}")

result: 0.126
answer: 0.080
situation: 0.079
scene: 0.025
response: 0.023


In [14]:
trainer.evaluate(tokenized_data["test"])

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 3.637598752975464,
 'eval_runtime': 58.2619,
 'eval_samples_per_second': 197.659,
 'eval_steps_per_second': 12.358,
 'epoch': 3.0}